# Accessing Data

In [ ]:
# Import Libraries
import pandas as pd
from datasets import load_dataset
import matplotlib.pyplot as plt

# Load Data
dataset = load_dataset('lukebarousse/data_jobs')
df = dataset['train'].to_pandas()

# Data Cleanup
df['job_posted_date'] = pd.to_datetime(df['job_posted_date'])

In [ ]:
# Utilise .loc to select slice columns by name
# Use .dropna to remove NaN values form salary_rate column
df.loc[:, 'salary_rate':'salary_hour_avg'].dropna(subset='salary_rate')

# Data Cleanup

In [ ]:
#Capture the median values
median_salary_year = df['salary_year_avg'].median()
median_salary_hour = df['salary_hour_avg'].median()

In [ ]:
#Create new DF of df_filled and replace nan values with median
df_filled = df.copy()
df_filled['salary_year_avg'] = df_filled['salary_year_avg'].fillna(median_salary_year)
df_filled['salary_hour_avg'] = df_filled['salary_hour_avg'].fillna(median_salary_hour)
df_filled['salary_year_avg']



In [ ]:
#Drop duplicates
df_unique = df_filled.drop_duplicates()

#We've dropped this many rows
print("rows dropped =", len(df_filled) - len(df_unique))

In [ ]:
#Can specify by columns, good to remove multiple jobs posted by the same company
df_unique = df_unique.drop_duplicates(subset=['job_title', 'company_name'])
#We've dropped this many rows
print("rows dropped =", len(df_filled) - len(df_unique))

# Pandas

Built on top of NumPy

##  Data Management

In [ ]:
#return a random sample of rows from a df (similar to head/tail)
#Can return the same answers by specifying a value to "random_state"
df_unique.sample(5, random_state=42)

In [ ]:
#By using the .copy() method we assign the new df a unique variable but transfer the data
df_original = dataset['train'].to_pandas()
df_altered = df_original.copy()

print(id(df_altered), id(df_original))

In [ ]:
#Without the .copy method the new df inherits the original ID, meaning any changes to df_copy impact df_original
df_copy = df_original
print(id(df_copy), id(df_original))

## Pivot Tables

In [ ]:
#create a pivot table by specifying the column(s) you want and the aggregation function
#size counts all rows in a column, count only counts non-NA values
df.pivot_table(index='job_title_short', aggfunc='size')

In [ ]:
#can create more complex pivots
df.pivot_table(values = ['salary_year_avg', 'salary_hour_avg'], index='job_title_short', aggfunc='mean')

In [ ]:
#Or even more complex pivots, and swap index/columns to make it longer
df.pivot_table(values = 'salary_year_avg', index='job_country' , columns='job_title_short', aggfunc='median')

In [ ]:
#Create an array of the top 6 countries in df
top_countries = df['job_country'].value_counts().head(6).index
top_countries

In [ ]:
#Assign the pivot table
df_job_country_salary = df.pivot_table(
    values='salary_year_avg',
    index='job_country',
    columns='job_title_short',
    aggfunc='median'
)


In [ ]:
#Filter pivot for only top countries
df_job_country_salary = df_job_country_salary.loc[top_countries]
df_job_country_salary

In [ ]:
#specify desired job titles for comparison
desired_jobs = ['Data Analyst','Data Engineer', 'Data Scientist']

#filter pivot for these roles
df_job_country_salary = df_job_country_salary[desired_jobs]

df_job_country_salary

In [ ]:
#plot 
df_job_country_salary.plot(kind='bar')
plt.ylabel('Median Salary ($)')
plt.xlabel('Country')
plt.xticks(rotation=45, ha='right')
plt.show()

## Index Management

In [ ]:
#check in index of a dataframe (first column)
df.index

#If we check the name, it is blank
df.index.name

#You can assign one
df.index.name = 'Job_Index'

#Inspect the data type
df.index.dtype

### reset index

In [ ]:
#When we subset a dataframe, the index values remain the same
df_usa = df[df['job_country']=='United States'].copy()

df_usa

In [ ]:
#Use .reset_index() to add a new column with a new index in numerical order, the old one is still preserved
df_usa.reset_index(inplace=True)
df_usa

### Set Index

In [ ]:
#Can assign a different column to be the index
df_usa.set_index('Job_Index', inplace=True)
df_usa

### Sort Index

In [ ]:
#useful when grouping or pivoting data, particularly when index doesn't have a column title but you still want to sort the data
median_pivot = df_usa.pivot_table(values='salary_year_avg', index='job_title_short', aggfunc=['median','min','max'])
median_pivot.sort_index()

## Exercise: Monthly Job Demand

In [ ]:
#filter data for UK
df_uk = df[df['job_country']=='United Kingdom']
df_uk

In [ ]:
#Add job post month
df_uk['job_posted_month'] = df_uk['job_posted_date'].dt.strftime('%B')
df_uk

In [ ]:
#create a pivot
uk_job_posts = df_uk.pivot_table(
    index='job_posted_month',
    columns='job_title_short',
    aggfunc='size'
)

uk_job_posts

In [ ]:
#Add a column for the numerical month
uk_job_posts.reset_index(inplace=True)
uk_job_posts['Month_no'] = pd.to_datetime(uk_job_posts['job_posted_month'], format='%B').dt.month
uk_job_posts

In [ ]:
#Sort values by numerical month
uk_job_posts.sort_values('Month_no', inplace=True)
#Index it according to the written month
uk_job_posts.set_index('job_posted_month', inplace=True)
#Drop the Month_no column
uk_job_posts.drop(columns='Month_no', inplace=True)
uk_job_posts

In [ ]:
#select for the three job titles we're interested in
top_uk_jobs = df_uk['job_title_short'].value_counts().head(3)
top_uk_jobs = top_uk_jobs.index.tolist()
top_uk_jobs


In [ ]:
#Filter our dataframe for the top three jobs and plot
uk_job_posts[top_uk_jobs].plot(kind='line')
plt.title('Monthly Posts for UK Tech Jobs')
plt.xlabel('2023')
plt.ylabel('Count')
plt.legend()
plt.show()

## Merge Data Frames

In [ ]:
#The two dataframes we're going to be merging
uk_job_posts
df_software_pivot = pd.read_csv("https://lukeb.co/software_csv", index_col='job_posted_month')
df_software_pivot

In [ ]:
#merge both dfs by job_posted_month
df_merged = uk_job_posts.merge(df_software_pivot, on='job_posted_month')
df_merged

In [ ]:
#Find the top 5 job postings across the whole dataframe
#We can indent the actions to make them easier to follow
top_5 = (
    df_merged.sum()
    .sort_values(ascending=False)
    .head(5)
    .index
    .tolist()
    )
top_5

In [ ]:
#Plot the data
df_merged[top_5].plot(kind='line')
plt.title('Top 5 Job Posts')
plt.xlabel('2023')
plt.ylabel('Job Post Count')
plt.ylim(0,20_000)
plt.legend()
plt.show()


## Concatenate Data Frames

In [ ]:
#Append a month column to df
df['job_posted_month'] = df['job_posted_date'].dt.strftime('%b')


In [ ]:
#list the months
months = df['job_posted_month'].unique()

In [ ]:
#Using dictionary comprehension to assign the months to a dictionary
dict_months = {month: df[df['job_posted_month'] == month] for month in months}

In [ ]:
#Can now access a dataframe for a specific month by calling the dictionary and key
dict_months['Dec']

In [ ]:
#Now we can concatenate these dataframes
#provide a list [] of dfs to concatenate, ignore_index removes the index column which would have duplicate values
df_Q1 = pd.concat([dict_months['Jan'], dict_months['Feb'], dict_months['Mar']], ignore_index=True)

#Check it has worked
df_Q1['job_posted_month'].value_counts().plot(kind='bar')

## Exporting Data Frames

* df.to_clipboard
* df.to_csv
* df.to_excel
* df.to_sql
* df.to_parquet
* df.to_pickle

In [ ]:
#To csv
df_Q1.head(10).to_csv('Quarter_One.csv')
pd.read_csv('Quarter_One.csv', index_col=0) #Add index_col=0 to remove unnecessary column 

In [ ]:
#To Excel
df_Q1.head(10).to_excel('Quarter_One.xlsx')

In [ ]:
#To sql
#from sqlalchemy import create_engine
# engine = create_engine('sqlite:///jobs.db')

# df_Q1.head(10).to_sql('Quarter_One', con=engine, if_exists='append', index=False)

In [ ]:
#Both good for large datasets
#To parquet 
df_Q1.head(10).to_parquet('Quarter_One.parquet')
#To pickle
df_Q1.head(10).to_pickle('Quarter_One.pkl')

## Applying Functions

In [ ]:
#Lots of nan values
df['salary_year_avg']

#Only show salary_year_avg data with values
df[pd.notna(df['salary_year_avg'])]['salary_year_avg']

#Create a filtered df
df_filtered = df[pd.notna(df['salary_year_avg'])].copy()

In [ ]:
#define a function for correcting salary for inflation
def projected_salary(salary):
    return salary * 1.03

#Apply it to the salary year avg
df_filtered['salary_year_avg'].apply(projected_salary)

#Create it as a new column 
df_filtered['inflated_year_salary'] = df_filtered['salary_year_avg'].apply(projected_salary)

df_filtered.head(5)

In [ ]:
df_filtered[['salary_year_avg', 'inflated_year_salary']].head(5)

In [ ]:
# We can write this quicker as a lambda function
#create a separate copy of the original data
df_anon = df[pd.notna(df['salary_year_avg'])].copy()

#Define a new column and apply the lambda function
df_anon['inflated_year_salary'] = df_anon['salary_year_avg'].apply(lambda salary: salary*1.03)

#view the result
df_anon[['salary_year_avg', 'inflated_year_salary']].head(5)

### Use Apply() in a real scenario

In [ ]:
#Can see that job_skills is formatted as a str
df['job_skills'][1]
type(df['job_skills'][1])

df_edit = df.copy()

In [ ]:
#Import ast and use literal_eval to format one cell as a list
import ast

ast.literal_eval(df_edit['job_skills'][1])

In [ ]:
#define a function to clean
def clean_list(skill_list):
    return ast.literal_eval(skill_list)

#Apply it
#df['job_skills'] = df['job_skills'].apply(clean_list)

#this causes an error due to the presence of nan values
df_edit[pd.isna(df_edit['job_skills'])].head(2)

In [ ]:
#Redefine the function to account for this 
def clean_list(skill_list):
    if pd.notna(skill_list):
        return ast.literal_eval(skill_list)

#Apply it
df_edit['job_skills'] = df_edit['job_skills'].apply(clean_list)


In [ ]:
#Now returns as type list
type(df_edit['job_skills'][2])

df_edit.head(10)

In [ ]:
#refresh the edited dataframe from the original
df_edit = df.copy()

#Can also structure it as a lambda function
df_edit['job_skills'] = df_edit['job_skills'].apply(lambda skill_list: ast.literal_eval(skill_list) if pd.notna(skill_list) else skill_list)


In [ ]:
df_edit.head(2)

## Use Apply() for rows

* Assume senior role salary inflation = 5 %
* Assume base salary inflation = 3 %

In [ ]:
#refresh the edited dataframe from the original
df_filtered = df[pd.notna(df['salary_year_avg'])].copy()
df_filtered.head(2)


In [ ]:
#Create the function
def projected_salary(row):
    if "Senior" in row['job_title_short']:
        return row['salary_year_avg'] * 1.05
    else:
        return row['salary_year_avg'] * 1.03

In [ ]:
#Because we are utilising information from multiple columns, we pass the whole df (df_filtered) to .apply, rather than a specific column
#We specify axis=1 because we are applying a function across rows, not down columns
df_filtered['salary_year_inflated'] = df_filtered.apply(projected_salary, axis=1)


In [ ]:
#Inspect the outcome
df_filtered[['job_title_short', 'salary_year_avg','salary_year_inflated']].sample(10,random_state=42)

## Explode

In [ ]:
#Using df_edit to explode the job_skills list
df_edit.info()
df_edit.head(5)

In [ ]:
#My attempt with no guidance of using .explode
#First subset job posts by the title Data Analyst, then explode the skills in these rows and only grab the top 10 
job_skills_counts = df_edit[df_edit['job_title_short'] == "Data Analyst"].explode(column='job_skills').value_counts('job_skills').head(10)


In [ ]:
#Plotting the data
job_skills_counts = job_skills_counts.sort_values(ascending=True)
job_skills_counts.plot(kind='barh')
plt.title('Top 10 Skills for Data Analysts')
plt.ylabel('Job Skills')
plt.xlabel('Job Posting Count')


In [ ]:
#The video groups by job title and skills
df_exploded = df_edit.explode('job_skills').copy()
df_exploded

In [ ]:
#Include reset_index to convert the series into a dataframe
df_skills_count = df_exploded.groupby(['job_title_short','job_skills']).size().reset_index(name='skill_count').copy()

#Sort it by the count column
df_skills_count = df_skills_count.sort_values(by='skill_count', ascending=False)

df_skills_count

In [ ]:
#Set variables for the filters
#Can set these to anything and alter the plot
title = 'Data Engineer'
top_skills = 20

#Apply the filters to get the final df
final_df = df_skills_count[df_skills_count['job_title_short'] == title].head(top_skills)

final_df
#Plotting the data
final_df.plot(kind='barh', x='job_skills', y='skill_count')
plt.gca().invert_yaxis() #Use this to invert the plotting order on yaxis
plt.title(f'Top {top_skills} Skills for {title}') #Using fstring for title to adapt it in response to variable changes
plt.ylabel('Job Skills')
plt.xlabel('Job Posting Count')
plt.legend().set_visible(False) #remove legend
plt.show()

## Exercise: Trending Skills Over Time

In [ ]:
#Create a column with the numerical month
df_skilltime = df_edit.copy()
df_skilltime['job_month_no'] = df_skilltime['job_posted_date'].dt.month
df_DA = df_skilltime[df_skilltime['job_title_short'] == 'Data Analyst']
df_DA.head(2)


In [ ]:
#Explode the job skills
df_explode = df_DA.explode('job_skills')

df_pivot = (df_explode.
    pivot_table(index = 'job_month_no', #pivot by the job title month
                columns = 'job_skills',  #set the skills as the column headers
                aggfunc='size', #set counts as the values
                fill_value=0)) #replace nan values with 0

df_pivot

In [ ]:
#Create a totals row
df_pivot.loc['Total'] = df_pivot.sum()
df_pivot

In [ ]:
#Sort the total values by size
df_pivot.loc['Total'].sort_values(ascending=False)

#Find the column headers for these values
df_pivot.loc['Total'].sort_values(ascending=False).index

In [ ]:
#Use this to order the columns by the totals row
df_pivot = df_pivot[df_pivot.loc['Total'].sort_values(ascending=False).index]


In [ ]:
#Drop the total row
df_pivot = df_pivot.drop('Total')
df_pivot

In [ ]:
df_pivot.iloc[:,0:5].plot(kind='line')
plt.title('Top 5 Job Skills for Data Analysts Per Month')
plt.xlabel('Month')
plt.ylabel('Count')

# Matplotlib

In [ ]:
#Using this df because the data is already formatted correctly
df_edit.info()


In [ ]:
#This is the layout for creating subplots using mpl
fig, ax = plt.subplots(1,1)

#passing the data we want into the plot
df['job_title_short'].value_counts().plot(kind='bar', ax = ax)

## Pandas notation for laying out plots

In [ ]:
fig, ax = plt.subplots(1,2)
#When plotting multiple plots, you need to specify the index of variable ax
df['job_title_short'].value_counts().plot(kind='barh', ax = ax[0])
df['job_schedule_type'].value_counts().head(3).plot(kind='barh', ax = ax[1])

#passing things to variable fig applies it to the whole figure
fig.tight_layout() #tight layout stops overlap of plot items

## Matplotlib Notation for the same plots

In [ ]:
#More verbose method, easier to use Pandas notation
fig, ax = plt.subplots(2,1)

ax[0].bar(df_edit['job_title_short'].value_counts().index, df_edit['job_title_short'].value_counts())
ax[1].bar(df_edit['job_schedule_type'].value_counts().head(3).index, df_edit['job_schedule_type'].value_counts().head(3))

plt.tight_layout
plt.show()

## Example: Counts of top skills in job postings

In [ ]:
#Include reset_index to convert the series into a dataframe
df_skills_count = df_exploded.groupby(['job_title_short','job_skills']).size().reset_index(name='skill_count').copy()

#Sort it by the count column
df_skills_count.sort_values(by='skill_count', ascending=False, inplace=True)

df_skills_count

In [ ]:
#specify the jobs we want to plot
titles = ['Data Scientist', 'Data Analyst' , 'Data Engineer']



In [ ]:
#Call the plot and specify the layout
fig, ax = plt.subplots(3,1)

#Within a for loop, use i to count through the list of tiles
for i, title in enumerate(titles):
    df_plot = df_skills_count[df_skills_count['job_title_short'] == title].head(5) #define the data which goes into the plot
    df_plot.plot(kind='barh', x='job_skills', y='skill_count', ax=ax[i], title=title) #specify the plot features and assign the index ax[i] so it counts through titles

#Specify preferences for all plots 
    ax[i].invert_yaxis() #invert plotting of yaxis so largest is on top
    ax[i].set_ylabel('') #remove yaxis label
    ax[i].legend().set_visible(False) #remove legend
    ax[i].set_xlim(0,120_000) #unify the xaxis
    
#Set the title for the whole figure
    fig.suptitle('Counts of Top Skills in Job Postings', fontsize=15)

#Call tight layout last to prevent overlaps
    fig.tight_layout()
    plt.show

## Pie Plots

In [ ]:
#Good for showing few values
df_edit['job_work_from_home'].value_counts().plot(kind = 'pie', 
                                                    startangle = 90, #can add this so values start from the top
                                                    autopct = '%1.2f%%') #% symbolises we're formatting the percentage, 1 is the minimum width, .2 for the number of decimals, f for floating point number, %% to show percentage symbol
plt.title('Work From Home Status')
plt.ylabel('')
plt.show()

In [ ]:
df_edit[['job_work_from_home', 'job_no_degree_mention','job_health_insurance']]

In [ ]:
#call the plot
fig, ax = plt.subplots(1,3)

#define a dictionary of the column names and the title we want to give it
dict_col = {
    'job_work_from_home': 'Work From Home',
    'job_no_degree_mention': 'Job Degree Req.',
    'job_health_insurance': 'Health Insurance Offered'
}

#Using a for loop to cycle through multiple plots
#because we're calling from a dictionary, we have to pass variables (col,title) as a tuple
#enumerate lets us cycle through the items of dict_col using i
for i, (col, title) in enumerate(dict_col.items()):
    ax[i].pie(df_edit[col].value_counts(), startangle=90, autopct='%1.2f%%', labels = ['False','True'])
    ax[i].set_title(title)
    
plt.show()


## Scatter Plots

In [ ]:
#subset by data analyst
df_da = df_edit[df_edit['job_title_short'] == 'Data Analyst'].copy()

#explode the skills
df_explode = df_da.explode('job_skills')



In [ ]:
#group by and apply functions to the df
skill_stats = df_explode.groupby('job_skills').agg( #multiple functions are being applied so we use agg
    skill_count = ('job_skills', 'count'), #skill count is the count of job skills column
    median_salary = ('salary_year_avg', 'median')
)

top_skill_stats = skill_stats.sort_values('skill_count', ascending=False).head(10)

In [ ]:
#plot the data
top_skill_stats.plot(kind='scatter', x='skill_count', y='median_salary')

#make it look nice
plt.xlabel('Count of Job Postings')
plt.ylabel('Median Salary ($)')
plt.title('Salary vs. Count of Job Postings for Top 10 Skills')
plt.tight_layout()    

#to add labels to the data, we need to loop through the index of the df 
for i, txt in enumerate(top_skill_stats.index):
    plt.text(top_skill_stats['skill_count'].iloc[i], top_skill_stats['median_salary'].iloc[i], txt) #use iloc to return the numerical values associated with i

plt.show()

# Advanced Customisation

## Line Plot

In [ ]:
#Explode the job skills
df_explode = df_DA.explode('job_skills')

df_pivot = (df_explode.
    pivot_table(index = 'job_month_no', #pivot by the job title month
                columns = 'job_skills',  #set the skills as the column headers
                aggfunc='size', #set counts as the values
                fill_value=0)) #replace nan values with 0

# Sort skills by count
df_pivot.loc['Total'] = df_pivot.sum()
df_pivot = df_pivot[df_pivot.loc['Total'].sort_values(ascending=False).index]
df_pivot = df_pivot.drop('Total')

#Sort months by number
df_pivot = df_pivot.reset_index()
df_pivot['job_posted_month'] = df_pivot['job_month_no'].apply(lambda x: pd.to_datetime(x, format = '%m').strftime('%b'))
df_pivot = df_pivot.set_index('job_posted_month')
df_pivot = df_pivot.drop(columns='job_month_no')

#A previous plot we did
df_pivot.iloc[:,0:5].plot(kind='line')
plt.title('Top 5 Job Skills for Data Analysts Per Month')
plt.xlabel('Month')
plt.ylabel('Count')

In [ ]:
#Upgrading the plot formatting
df_pivot.iloc[:,0:5].plot(
    kind='line', 
    linewidth=2,
    linestyle='-', #options: '-', '--', '-.', ':'
    colormap='viridis', #options: 'viridis', 'plasma', 'magma', 'inferno', 'cividis'
    markersize=6, 
    marker='s', #options: 'o', 'x', '+', '*', 's', ',', '.', '1', '2', '3', '4'
    figsize=(10,5)    
    )

plt.xlabel('2023')
plt.ylabel('Count')
plt.title('Top 5 Job Skills for Data Analysts Per Month')

plt.show()

## Scatter Plot

In [ ]:
#Original Plot
skill_no = 20
top_skill_stats = skill_stats.sort_values('skill_count', ascending=False).head(skill_no)

#plot the data
top_skill_stats.plot(kind='scatter', x='skill_count', y='median_salary')

#make it look nice
plt.xlabel('Count of Job Postings')
plt.ylabel('Median Salary ($)')
plt.title('Salary vs. Count of Job Postings for Top 10 Skills')
plt.tight_layout()    

#to add labels to the data, we need to loop through the index of the df 
for i, txt in enumerate(top_skill_stats.index):
    plt.text(top_skill_stats['skill_count'].iloc[i], top_skill_stats['median_salary'].iloc[i], txt) #use iloc to return the numerical values associated with i

plt.show()

In [ ]:
from adjustText import adjust_text

#plot the data
top_skill_stats.plot(kind='scatter', x='skill_count', y='median_salary')

#create an empty list
texts = []
#to add labels to the data, we need to loop through the index of the df 
for i, txt in enumerate(top_skill_stats.index):
    texts.append(plt.text(top_skill_stats['skill_count'].iloc[i], top_skill_stats['median_salary'].iloc[i], txt)) #append the empty list with the text coordinate data
    
adjust_text(texts,  #pass the list of text info from texts
            arrowprops=dict(arrowstyle='-',  #set the arrow style
                            color = 'b',  #set arrow colour
                            lw=0.5)) #set line width


ax = plt.gca() #assign gca function to ax

#make it look nice
plt.xlabel('Count of Job Postings')
plt.ylabel('Median Salary ($)')
plt.title('Salary vs. Count of Job Postings for Top 10 Skills')
plt.tight_layout()  

ax.yaxis.set_major_formatter( #call ax that we assigned above
    plt.FuncFormatter( #this allows us to call func formatter 
        lambda y, pos: f'${int(y/1000)}K' #format the yaxis labels as an f string inside of a lambda function, have to pass the pos argument even if unused
        )
    )

#Do the same to make the x axis labels comma separated
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{int(x):,}'))

plt.show()

# Statistical Plots

## Histograms

In [ ]:
#subset the data for US data analysts
us_da = df_edit[(df_edit['job_title_short'] == 'Data Analyst') &(df_edit['job_country'] == 'United States')].copy()
us_da.head(3)

In [ ]:
#Simple histogram
us_da['salary_year_avg'].plot(kind='hist')

In [ ]:
#Customised Histogram
us_da['salary_year_avg'].plot(kind='hist', bins=40, edgecolor = 'black')

plt.xlim(0,250_000)

ax = plt.gca()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))

plt.xlabel('Annual Salary ($)')
plt.ylabel('Number of Jobs')
plt.title('Distribution of United States Annual Data Analyst Salaries')

## Box Plots

In [ ]:
df_usa2 = df_usa.dropna(subset=['salary_year_avg']).copy()
df_usa2.head(3)

In [ ]:
#Matplotlib
#plt.boxplot(df_usa2['salary_year_avg'])

#Pandas
df_usa2['salary_year_avg'].plot(kind='box', vert=False)


In [ ]:
#Create the multiplot
fig, ax = plt.subplots(2,1)
fig.tight_layout(h_pad=1)

#Histogram
df_usa2['salary_year_avg'].plot(kind='hist',ax=ax[0],bins=30, edgecolor='black')
ax[0].set_title('Salary Distributions for Data Analyst in the US')
ax[0].set_xlabel('')
ax[0].set_ylabel('Frequency')
ax[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))

#Histogram Lines
ax[0].axvline(df_usa2['salary_year_avg'].median(), color='red', linestyle='--')
ax[0].axvline(df_usa2['salary_year_avg'].quantile(0.25), color='gold', linestyle='--')
ax[0].axvline(df_usa2['salary_year_avg'].quantile(0.75), color='gold', linestyle='--')

#Boxplot
df_usa2['salary_year_avg'].plot(kind='box', ax=ax[1], vert=False)
ax[1].set_yticklabels('')
ax[1].set_xlabel('Average Annual Salary ($)')
ax[1].set_ylabel('')
ax[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))

#Boxplot Lines
ax[1].axvline(df_usa2['salary_year_avg'].median(), color='red', linestyle='--')
ax[1].axvline(df_usa2['salary_year_avg'].quantile(0.25), color='gold', linestyle='--')
ax[1].axvline(df_usa2['salary_year_avg'].quantile(0.75), color='gold', linestyle='--')

plt.show()

In [ ]:
job_titles = ['Data Analyst', 'Data Engineer', 'Data Scientist']


df_usa3 = df_edit[
    (df_edit['job_title_short'].isin(job_titles)) &
    (df_edit['job_country'] == 'United States')
    ].copy()

df_usa3 = df_usa3.dropna(subset='salary_year_avg')

df_usa3.head(3)

In [ ]:
#list comprehension to index the salary year avg values for each job in job_titles
job_list = [df_usa3[df_usa3['job_title_short'] == job_title]['salary_year_avg'] for job_title in job_titles]
job_list[0]

In [ ]:
plt.boxplot(job_list, tick_labels=job_titles, orientation='horizontal')

plt.title('Salary Distribution in the United States')
plt.xlabel('Annual Salary ($)')
plt.ylabel('')

ax = plt.gca()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000):,}K')) #format the xaxis

plt.xlim(0,600_000) #filter out some outliers and focus more on the majority of data

plt.show()

# Exercise: Skill Pay Analysis

In [ ]:
#subset data
df_usa4 = df_edit[(df_edit['job_title_short'] == 'Data Analyst') & (df_edit['job_country'] == 'United States')].dropna(subset='salary_year_avg').copy()
df_usa4.head(3)

In [ ]:
#Explode job
df4_explode = df_usa4.explode('job_skills')
df4_explode.head(3)


In [ ]:
#aggregate counts of job skills and their associated median salary
df4_group = df4_explode.groupby('job_skills')['salary_year_avg'].agg(['count', 'median'])
df4_group.head(3)

In [ ]:
#need to create 2 new dfs, one with the top 10 skill counts, and another with the top 10 median salaries
top_skills = df4_group.sort_values(by='count',ascending=False).head(10).sort_values(by='median', ascending=False)
top_pay = df4_group.sort_values(by='median',ascending=False).head(10)


In [ ]:
#Plot the data
fig, ax = plt.subplots(2,1)
fig.tight_layout()

#skills plot
top_skills[::-1].plot(kind='barh', y='median',  ax=ax[0],legend=False) #Call the df in reverse order as a way to invert yaxis plotting

#Pay plot
top_pay.plot(kind='barh', y='median',ax=ax[1],legend=False)

#customise skills
ax[0].set_title('Top 10 In Demand Skill for Data Analysts')
ax[0].set_ylabel('')
ax[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _ : f'${int(x/1_000)}K'))
ax[0].set_xlim(ax[1].get_xlim()) #can use this to automate xlims, but must come after plotting

#customise pay
ax[1].set_title('Top 10 Paying Skills for Data Analysts')
ax[1].invert_yaxis()
ax[1].set_ylabel('')
ax[1].set_xlabel('Median Annual Salary ($)')
ax[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _ : f'${int(x/1_000)}K'))

fig.tight_layout()
plt.show()


# Seaborn

Built on top of Matplotlib, specialised for visualisations

In [ ]:
#Install and import seaborn
import seaborn as sns

## Bar Charts

In [ ]:
#using pay and skills data from earlier, modify the plot using Seaborn

#Plot the data
fig, ax = plt.subplots(2,1)
fig.tight_layout()

#MatPlotLib code
#top_pay.plot(kind='barh', y='median',ax=ax[0],legend=False)
#top_skills[::-1].plot(kind='barh', y='median',  ax=ax[1],legend=False) #Call the df in reverse order as a way to invert yaxis plotting

#Seaborn code
sns.barplot(top_pay[::-1], #provide data 
            x='median',  #set x
            y=top_pay.index, #set y
            ax=ax[0], #ax id
            hue='median', #data for colouring 
            palette='dark:b_r', #palette (dark:b), _r reverses order of colour plotting (light->dark/dark->light)
            legend=False) #remove legend
sns.barplot(top_skills,x='median', y=top_skills.index, ax=ax[1], hue='median', palette='dark:r_r', legend=False)

#customise pay
ax[0].set_title('Top 10 Paying Skills for Data Analysts')
ax[0].invert_yaxis()
ax[0].set_ylabel('')
ax[0].set_xlabel('Median Annual Salary ($)')
ax[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _ : f'${int(x/1_000)}K'))

#customise skills
ax[1].set_title('Top 10 In Demand Skill for Data Analysts')
ax[1].set_ylabel('')
ax[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _ : f'${int(x/1_000)}K'))
ax[1].set_xlim(ax[0].get_xlim()) #can use this to automate xlims, but must come after plotting

#Can set themes
sns.set_theme(style='ticks')

fig.tight_layout()
plt.show()


## Histograms

In [ ]:
#Old histogram Plot
#us_da['salary_year_avg'].plot(kind='hist')

#New histogram with Seaborn
sns.histplot(us_da, x = 'salary_year_avg')



In [ ]:
#Can also plot the data with a distribution plot for smoother lines

sns.displot(us_da, 
            x='salary_year_avg',
            kind = 'kde',
            fill=True)
plt.title('Distribution of US Data Analyst Annual Salaries')
plt.xlabel('Annual Salary')
plt.ylabel('Number of Jobs')
ax = plt.gca()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))


## Box Plots

In [ ]:
#Old Boxplot
plt.boxplot(job_list, tick_labels=job_titles, orientation='horizontal')

plt.title('Salary Distribution in the United States')
plt.xlabel('Annual Salary ($)')
plt.ylabel('')

ax = plt.gca()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000):,}K')) #format the xaxis

plt.xlim(0,600_000) #filter out some outliers and focus more on the majority of data

plt.show()

In [ ]:
#Seaborn Boxplot

sns.boxplot(data=df_usa2[df_usa2['job_title_short'].isin(job_titles)],
            x = 'salary_year_avg',
            y = 'job_title_short')
plt.title('Salary Distribution in the United States')
plt.xlabel('Annual Salary ($)')
plt.ylabel('')
ax = plt.gca()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000):,}K')) #format the xaxis
plt.xlim(0,600_000) #filter out some outliers and focus more on the majority of data
plt.show()
